# Validación

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from funciones_modelado import metricas

In [2]:
b0    = pd.read_parquet("predicciones/b0.parquet")
b1    = pd.read_parquet("predicciones/b1.parquet")
b2    = pd.read_parquet("predicciones/b2.parquet")
ridge = pd.read_parquet("predicciones/ridge_valid.parquet")
rf    = pd.read_parquet("predicciones/rf_valid.parquet")
lgbm  = pd.read_parquet("predicciones/lgbm_valid.parquet")

for nombre, d in {"b0": b0, "b1": b1, "b2": b2, "ridge": ridge, "rf": rf, "lgbm": lgbm}.items():
    print(f"{nombre:6} | {d.shape} | {list(d.columns)}")
# las predicciones de B0, B1 y B2 contienen el total de instancias del df, pero train tiene NaN en sus columnas

b0     | (14223, 5) | ['IDMOTask', 'split', 'mes_orden', 'ratio', 'pred_B0']
b1     | (14223, 6) | ['IDMOTask', 'split', 'mes_orden', 'ratio', 'pred_B1_mediana', 'pred_B1_media']
b2     | (14223, 8) | ['IDMOTask', 'split', 'mes_orden', 'ratio', 'pred_B2_familia_mediana', 'pred_B2_familia_media', 'pred_B2_maquina_mediana', 'pred_B2_maquina_media']
ridge  | (2887, 5) | ['IDMOTask', 'mes_orden', 'ratio', 'pred_ridge_ratio', 'pred_ridge_log']
rf     | (2887, 5) | ['IDMOTask', 'mes_orden', 'ratio', 'pred_rf_ratio', 'pred_rf_log']
lgbm   | (2887, 5) | ['IDMOTask', 'mes_orden', 'ratio', 'pred_lgbm_ratio', 'pred_lgbm_log']


## Ridge

In [7]:
valid = ridge[["IDMOTask", "mes_orden", "ratio", "pred_ridge_ratio", "pred_ridge_log"]]

for d in [rf, lgbm, b0, b1, b2]:
    cols_pred = [c for c in d.columns if c.startswith("pred_")]
    valid = valid.merge(d[["IDMOTask"] + cols_pred], on="IDMOTask", how="left")

print(valid.shape)
print(valid.isna().sum().sum())

columnas_orden = ["IDMOTask", "mes_orden", "ratio", "pred_B0", "pred_B1_mediana", "pred_B1_media", "pred_B2_familia_mediana", "pred_B2_familia_media", "pred_B2_maquina_mediana", "pred_B2_maquina_media", "pred_ridge_ratio", "pred_ridge_log", "pred_rf_ratio", "pred_rf_log", "pred_lgbm_ratio", "pred_lgbm_log"]
valid = valid[columnas_orden]

(2887, 16)
0


In [8]:
valid.filter(like="pred_").describe().loc[["mean", "50%"]].round(2)

,pred_B0,pred_B1_mediana,pred_B1_media,pred_B2_familia_mediana,pred_B2_familia_media,pred_B2_maquina_mediana,pred_B2_maquina_media,pred_ridge_ratio,pred_ridge_log,pred_rf_ratio,pred_rf_log,pred_lgbm_ratio,pred_lgbm_log
mean,1.0,1.46,1.87,1.47,1.88,1.50,1.89,1.88,1.57,1.88,1.58,1.87,1.59
50%,1.0,1.45,1.87,1.56,1.99,1.54,1.95,1.88,1.52,1.81,1.53,1.77,1.54


Comparativa de validaciones:

In [17]:
cols_pred = [c for c in valid.columns if c.startswith("pred_")]
meses = sorted(valid["mes_orden"].unique())

filas = []
for mes in meses:
    datos = valid[valid["mes_orden"] == mes]
    for col in cols_pred:
        m = metricas(datos["ratio"], datos[col])
        filas.append({"mes": mes, "prediccion": col, "MAE": m["MAE"], "RMSE": m["RMSE"], "n": m["n"]})

comparativa = pd.DataFrame(filas)
tabla_MAE  = comparativa.pivot(index="prediccion", columns="mes", values="MAE")
tabla_RMSE = comparativa.pivot(index="prediccion", columns="mes", values="RMSE")

tabla_MAE.index.name = "prediccion_MAE"
tabla_RMSE.index.name = "prediccion_RMSE"

tabla_MAE["media_folds"]  = tabla_MAE.mean(axis=1)
tabla_RMSE["media_folds"] = tabla_RMSE.mean(axis=1)

En la tabla de MAE los baselines B1 y B2 usan la mediana; en la de RMSE, la media. Cada métrica se compara con la versión del baseline que la minimiza

> Para los baselines, el MAE se calcula comparando la predicción basada en la **mediana** con el ratio real, y el RMSE comparando la predicción basada en la **media** con el ratio real.

In [19]:
# filtrar los modelos para que solo muestren sú metrica
     # MAE - mediana
     # RMSE - media 

modelos = ["pred_ridge_ratio", "pred_ridge_log", "pred_rf_ratio", "pred_rf_log",
           "pred_lgbm_ratio", "pred_lgbm_log"]

# Cada métrica con la versión de baseline que la minimiza
cols_MAE  = modelos + ["pred_B0", "pred_B1_mediana", "pred_B2_familia_mediana", "pred_B2_maquina_mediana"]
cols_RMSE = modelos + ["pred_B0", "pred_B1_media",   "pred_B2_familia_media",   "pred_B2_maquina_media"]

nombres = {"pred_B0": "B0",
           "pred_B1_mediana": "B1", "pred_B1_media": "B1",
           "pred_B2_familia_mediana": "B2-familia", "pred_B2_familia_media": "B2-familia",
           "pred_B2_maquina_mediana": "B2-máquina", "pred_B2_maquina_media": "B2-máquina"}

tabla_MAE_final  = tabla_MAE.loc[cols_MAE].rename(index=nombres).sort_values("media_folds")
tabla_RMSE_final = tabla_RMSE.loc[cols_RMSE].rename(index=nombres).sort_values("media_folds")

display(tabla_MAE_final.round(3))
display(tabla_RMSE_final.round(3))

mes,2026-02,2026-03,2026-04,media_folds
prediccion_MAE,,,,
pred_lgbm_log,0.895,0.797,0.812,0.835
pred_rf_log,0.911,0.815,0.809,0.845
pred_lgbm_ratio,0.901,0.813,0.828,0.847
pred_ridge_log,0.923,0.810,0.823,0.852
pred_rf_ratio,0.921,0.833,0.826,0.860
pred_ridge_ratio,0.938,0.832,0.844,0.872
B2-máquina,0.983,0.888,0.928,0.933
B2-familia,0.993,0.897,0.943,0.944
B1,1.026,0.926,0.971,0.974


mes,2026-02,2026-03,2026-04,media_folds
prediccion_RMSE,,,,
pred_lgbm_ratio,1.294,1.181,1.197,1.224
pred_rf_ratio,1.312,1.209,1.196,1.239
pred_ridge_ratio,1.322,1.201,1.209,1.244
pred_lgbm_log,1.369,1.247,1.270,1.296
pred_rf_log,1.386,1.272,1.267,1.308
pred_ridge_log,1.394,1.263,1.279,1.312
B2-máquina,1.425,1.315,1.342,1.361
B2-familia,1.450,1.332,1.365,1.382
B1,1.489,1.371,1.408,1.423


#### Comparativa en validación (3 folds)

Los MAE de los baselines coinciden con los obtenidos en `10.1_Baselines`, por lo que la unión de predicciones es correcta.

**MAE** (baselines con mediana):
- Los 6 modelos obtienen menor MAE que todos los baselines en los 3 folds, incluido B2-máquina, el más exigente (0.933 de media).
- Para cada modelo, la versión **log(ratio)** tiene menor MAE que la versión ratio en los 3 folds.
- El menor MAE medio es el de **LightGBM-log (0.835)**, seguido de RF-log (0.845) y LightGBM-ratio (0.847). El orden entre modelos no se mantiene en todos los folds: en abril RF-log (0.809) queda por delante de LightGBM-log (0.812).

**RMSE** (baselines con media):
- Los 6 modelos obtienen menor RMSE que todos los baselines en los 3 folds (B2-máquina-media: 1.361 de media).
- Para cada modelo, la versión **ratio** tiene menor RMSE que la versión log en los 3 folds.
- El menor RMSE medio es el de **LightGBM-ratio (1.224)**.

**En todos los casos** febrero es el fold con mayor error.

## skill score